2.1 — Setup iniziale e import delle librerie

In questo notebook si procede all’arricchimento del dataset master dei box office domestic con l’identificativo univoco IMDb (imdb_id).
L’operazione viene effettuata tramite interrogazione dell’API OMDb, utilizzando una procedura controllata, riproducibile e documentata.

In [1]:
import os
import time
import pandas as pd
import requests


2.2 — Caricamento del dataset master dei box office

Si importa il file boxoffice_master_domestic.csv, prodotto nel notebook precedente.
Questo file rappresenta la base dati consolidata su cui verrà effettuato il collegamento con IMDb.

In [2]:
MASTER_PATH = "data/boxoffice_master_domestic.csv"
boxoffice_master = pd.read_csv(MASTER_PATH)

print("Shape master:", boxoffice_master.shape)
boxoffice_master.head(5)


Shape master: (4100, 11)


,box_office_year,rank,title,release_date,release_year,distributor,genre,domestic_gross_raw,domestic_box_office_gross,tickets_sold_raw,tickets_sold
0,1980,1,Star Wars Ep. V: The Empire Strikes Back,"May 21, 1980",1980,20th Century Fox,Adventure,"$181,353,855",181353855,"67,417,790",67417790.0
1,1980,2,Stir Crazy,"Dec 12, 1980",1980,Columbia,Black Comedy,"$101,300,000",101300000,"37,657,992",37657992.0
2,1980,3,Kramer vs. Kramer,"Dec 19, 1979",1979,NaN,Drama,"$98,982,763",98982763,"36,796,566",36796566.0
3,1980,4,Airplane!,"Jul 4, 1980",1980,Paramount Pictures,Comedy,"$83,453,539",83453539,"31,023,620",31023620.0
4,1980,5,Any Which Way You Can,"Dec 17, 1980",1980,Warner Bros.,Comedy,"$70,687,344",70687344,"26,277,823",26277823.0


2.3 — Lettura e verifica della OMDb API key

La chiave OMDb è gestita come variabile d’ambiente (OMDB_API_KEY) per motivi di sicurezza e riproducibilità.
Prima di procedere con qualunque interrogazione massiva, si verifica che la chiave sia correttamente caricata.

In [3]:
OMDB_API_KEY = os.environ.get("OMDB_API_KEY")
print("OMDB_API_KEY presente?", OMDB_API_KEY is not None)

if not OMDB_API_KEY:
    raise ValueError("OMDB_API_KEY non trovata. Inseriscila in Deepnote come Environment Variable.")


OMDB_API_KEY presente? True


2.4 — Test preliminare dell’API OMDb

Prima dell’esecuzione su larga scala, si effettua un test puntuale su un titolo noto, al fine di verificare la corretta risposta dell’API e l’effettiva validità della chiave.

In [4]:
params = {
    "apikey": OMDB_API_KEY,
    "t": "Shrek 2",
    "y": "2004",
    "type": "movie"
}

r = requests.get("http://www.omdbapi.com/", params=params, timeout=20)

print("HTTP status:", r.status_code)
print("Response body:", r.text[:300])

if '"Response":"True"' not in r.text:
    raise RuntimeError("Test OMDb fallito: la chiave non risponde correttamente.")


HTTP status: 200
Response body: {"Title":"Shrek 2","Year":"2004","Rated":"PG","Released":"19 May 2004","Runtime":"93 min","Genre":"Animation, Adventure, Comedy","Director":"Andrew Adamson, Kelly Asbury, Conrad Vernon","Writer":"William Steig, Andrew Adamson, Joe Stillman","Actors":"Mike Myers, Eddie Murphy, Cameron Diaz","Plot":"S


2.5 — Costruzione delle chiavi uniche per il lookup IMDb

Per ridurre il numero di chiamate all’API, il lookup viene effettuato una sola volta per ciascuna combinazione univoca di titolo e anno di uscita (title, release_year).
Questa tabella di chiavi costituisce l’insieme minimo delle interrogazioni necessarie.

In [5]:
lookup_df = boxoffice_master[["title", "release_year"]].copy()

lookup_df["release_year"] = pd.to_numeric(
    lookup_df["release_year"], errors="coerce"
).astype("Int64")

lookup_df = lookup_df.drop_duplicates(
    subset=["title", "release_year"]
).reset_index(drop=True)

print("Chiavi uniche per lookup:", lookup_df.shape)
lookup_df.head(10)


Chiavi uniche per lookup: (3797, 2)


,title,release_year
0,Star Wars Ep. V: The Empire Strikes Back,1980
1,Stir Crazy,1980
2,Kramer vs. Kramer,1979
3,Airplane!,1980
4,Any Which Way You Can,1980
5,Private Benjamin,1980
6,Coal Miner's Daughter,1980
7,Smokey and the Bandit II,1980
8,The Blues Brothers,1980
9,Ordinary People,1980


2.6 — Definizione della funzione di interrogazione OMDb

Si definisce una funzione dedicata all’interrogazione dell’API OMDb.
La funzione restituisce l’identificativo IMDb quando disponibile e registra eventuali errori o mancate corrispondenze, consentendo una successiva analisi dei casi non risolti.

In [6]:
OMDB_URL = "http://www.omdbapi.com/"

def omdb_lookup_imdb_id(title, year, api_key):
    try:
        params = {
            "apikey": api_key,
            "t": str(title).strip(),
            "y": str(int(year)) if pd.notna(year) else None,
            "type": "movie"
        }
        params = {k: v for k, v in params.items() if v is not None}

        r = requests.get(OMDB_URL, params=params, timeout=20)
        data = r.json()

        if data.get("Response") == "True":
            return {
                "imdb_id": data.get("imdbID", pd.NA),
                "omdb_title": data.get("Title", pd.NA),
                "omdb_year": data.get("Year", pd.NA),
                "response": True,
                "error": pd.NA
            }
        else:
            return {
                "imdb_id": pd.NA,
                "omdb_title": pd.NA,
                "omdb_year": pd.NA,
                "response": False,
                "error": data.get("Error", "Unknown error")
            }
    except Exception as e:
        return {
            "imdb_id": pd.NA,
            "omdb_title": pd.NA,
            "omdb_year": pd.NA,
            "response": False,
            "error": str(e)
        }


2.7 — Interrogazione OMDb su larga scala con salvataggio incrementale

L’estrazione degli identificativi IMDb viene eseguita su tutte le chiavi uniche.
Per garantire robustezza e riproducibilità, i risultati vengono salvati progressivamente in un file di checkpoint, consentendo la ripresa dell’elaborazione in caso di interruzioni.

In [7]:
CHECKPOINT_PATH = "data/omdb_lookup_checkpoint.csv"
SAVE_EVERY = 50
SLEEP_SECONDS = 0.25

if os.path.exists(CHECKPOINT_PATH):
    done = pd.read_csv(CHECKPOINT_PATH)
    done_keys = set(zip(done["title"], done["release_year"]))
    print("Checkpoint trovato:", len(done))
else:
    done = pd.DataFrame(columns=[
        "title","release_year","imdb_id",
        "omdb_title","omdb_year","response","error"
    ])
    done_keys = set()
    print("Nessun checkpoint: avvio da zero.")

to_do = lookup_df[
    ~lookup_df.apply(lambda r: (r["title"], r["release_year"]) in done_keys, axis=1)
].copy()

print("Record da processare:", to_do.shape[0])

results = []

for i, row in to_do.iterrows():
    res = omdb_lookup_imdb_id(
        title=row["title"],
        year=row["release_year"],
        api_key=OMDB_API_KEY
    )

    results.append({
        "title": row["title"],
        "release_year": row["release_year"],
        **res
    })

    time.sleep(SLEEP_SECONDS)

    if len(results) % SAVE_EVERY == 0:
        done = pd.concat([done, pd.DataFrame(results)], ignore_index=True)
        done.to_csv(CHECKPOINT_PATH, index=False)
        print("Salvati record:", len(done))
        results = []

if results:
    done = pd.concat([done, pd.DataFrame(results)], ignore_index=True)
    done.to_csv(CHECKPOINT_PATH, index=False)

print("Lookup completato. Totale record:", len(done))
done.head(10)


Checkpoint trovato: 3797
Record da processare: 0
Lookup completato. Totale record: 3797


,title,release_year,imdb_id,omdb_title,omdb_year,response,error
0,Star Wars Ep. V: The Empire Strikes Back,1980,NaN,NaN,NaN,False,Movie not found!
1,Stir Crazy,1980,tt0081562,Stir Crazy,1980.0,True,NaN
2,Kramer vs. Kramer,1979,tt0079417,Kramer vs. Kramer,1979.0,True,NaN
3,Airplane!,1980,tt0080339,Airplane!,1980.0,True,NaN
4,Any Which Way You Can,1980,tt0080377,Any Which Way You Can,1980.0,True,NaN
5,Private Benjamin,1980,tt0081375,Private Benjamin,1980.0,True,NaN
6,Coal Miner's Daughter,1980,tt0080549,Coal Miner's Daughter,1980.0,True,NaN
7,Smokey and the Bandit II,1980,tt0081529,Smokey and the Bandit II,1980.0,True,NaN
8,The Blues Brothers,1980,tt0080455,The Blues Brothers,1980.0,True,NaN
9,Ordinary People,1980,tt0081283,Ordinary People,1980.0,True,NaN


2.8 — Analisi dei casi non risolti

Si isolano i film per i quali non è stato possibile ottenere un identificativo IMDb.
Questi casi vengono salvati in un file separato, utile per eventuali normalizzazioni o verifiche manuali.

In [8]:
lookup_enriched = pd.read_csv(CHECKPOINT_PATH)

not_found = lookup_enriched[lookup_enriched["imdb_id"].isna()].copy()
print("Casi non risolti:", not_found.shape[0])

LOG_PATH = "data/omdb_not_found_log.csv"
not_found.to_csv(LOG_PATH, index=False)

not_found.head(20)


Casi non risolti: 205


,title,release_year,imdb_id,omdb_title,omdb_year,response,error
0,Star Wars Ep. V: The Empire Strikes Back,1980,NaN,NaN,NaN,False,Movie not found!
23,Caligula,1980,NaN,NaN,NaN,False,Movie not found!
28,In Search of Historic Jesus,1980,NaN,NaN,NaN,False,Movie not found!
70,Mad Max,1980,NaN,NaN,NaN,False,Movie not found!
71,The Big Brawl,1980,NaN,NaN,NaN,False,Movie not found!
74,How to Beat the High Co$t of Living,1980,NaN,NaN,NaN,False,Movie not found!
80,Nine to Five,1980,NaN,NaN,NaN,False,Movie not found!
89,In God We Tru$t,1980,NaN,NaN,NaN,False,Movie not found!
93,Willie and Phil,1980,NaN,NaN,NaN,False,Movie not found!
101,Superman II,1981,NaN,NaN,NaN,False,Movie not found!


2.9 — Merge degli IMDb ID nel dataset master

I risultati del lookup vengono riagganciati al dataset master tramite una join sulle variabili title e release_year.
La scelta di una join sinistra garantisce la conservazione di tutte le osservazioni originali.

In [9]:
boxoffice_master["release_year"] = pd.to_numeric(
    boxoffice_master["release_year"], errors="coerce"
).astype("Int64")

master_enriched = boxoffice_master.merge(
    lookup_enriched[["title","release_year","imdb_id","response","error"]],
    on=["title","release_year"],
    how="left"
)

print("Master enriched shape:", master_enriched.shape)
master_enriched.head(5)


Master enriched shape: (4100, 14)


,box_office_year,rank,title,release_date,release_year,distributor,genre,domestic_gross_raw,domestic_box_office_gross,tickets_sold_raw,tickets_sold,imdb_id,response,error
0,1980,1,Star Wars Ep. V: The Empire Strikes Back,"May 21, 1980",1980,20th Century Fox,Adventure,"$181,353,855",181353855,"67,417,790",67417790.0,NaN,False,Movie not found!
1,1980,2,Stir Crazy,"Dec 12, 1980",1980,Columbia,Black Comedy,"$101,300,000",101300000,"37,657,992",37657992.0,tt0081562,True,NaN
2,1980,3,Kramer vs. Kramer,"Dec 19, 1979",1979,NaN,Drama,"$98,982,763",98982763,"36,796,566",36796566.0,tt0079417,True,NaN
3,1980,4,Airplane!,"Jul 4, 1980",1980,Paramount Pictures,Comedy,"$83,453,539",83453539,"31,023,620",31023620.0,tt0080339,True,NaN
4,1980,5,Any Which Way You Can,"Dec 17, 1980",1980,Warner Bros.,Comedy,"$70,687,344",70687344,"26,277,823",26277823.0,tt0080377,True,NaN


2.10 — Costruzione del file metadata.csv finale

Si produce il file metadata.csv, riducendo il dataset alle variabili essenziali per le analisi successive e per il collegamento con ulteriori fonti (es. sottotitoli).

Schema finale:
rank – imdb_id – title – box_office_year – release_year – distributor – domestic_box_office_gross – tickets_sold.

In [10]:
metadata = master_enriched[
    ["rank","imdb_id","title","box_office_year",
     "release_year","distributor",
     "domestic_box_office_gross","tickets_sold"]
].copy()

metadata["rank"] = pd.to_numeric(metadata["rank"], errors="coerce").astype("Int64")
metadata["box_office_year"] = pd.to_numeric(metadata["box_office_year"], errors="coerce").astype("Int64")
metadata["release_year"] = pd.to_numeric(metadata["release_year"], errors="coerce").astype("Int64")
metadata["domestic_box_office_gross"] = pd.to_numeric(
    metadata["domestic_box_office_gross"], errors="coerce"
).astype("Int64")
metadata["tickets_sold"] = pd.to_numeric(metadata["tickets_sold"], errors="coerce").astype("Int64")

OUT_PATH = "data/metadata.csv"
metadata.to_csv(OUT_PATH, index=False)

print("Salvato:", OUT_PATH)
metadata.head(10)


Salvato: data/metadata.csv


,rank,imdb_id,title,box_office_year,release_year,distributor,domestic_box_office_gross,tickets_sold
0,1,NaN,Star Wars Ep. V: The Empire Strikes Back,1980,1980,20th Century Fox,181353855,67417790
1,2,tt0081562,Stir Crazy,1980,1980,Columbia,101300000,37657992
2,3,tt0079417,Kramer vs. Kramer,1980,1979,NaN,98982763,36796566
3,4,tt0080339,Airplane!,1980,1980,Paramount Pictures,83453539,31023620
4,5,tt0080377,Any Which Way You Can,1980,1980,Warner Bros.,70687344,26277823
5,6,tt0081375,Private Benjamin,1980,1980,Warner Bros.,69847348,25965556
6,7,tt0080549,Coal Miner's Daughter,1980,1980,Universal,67182787,24975013
7,8,tt0081529,Smokey and the Bandit II,1980,1980,Universal,66132626,24584619
8,9,tt0080455,The Blues Brothers,1980,1980,Universal,57229890,21275052
9,10,tt0081283,Ordinary People,1980,1980,Paramount Pictures,52302978,19443486


In [11]:
coverage = metadata["imdb_id"].notna().mean()
coverage


0.9446341463414634

In [12]:
import pandas as pd
import re, unicodedata
from pathlib import Path

META_PATH = "data/metadata.csv"
m = pd.read_csv(META_PATH)

# --- normalizzazione (coerente e stabile) ---
def norm_title(s):
    s = "" if pd.isna(s) else str(s)
    s = unicodedata.normalize("NFKD", s)
    s = "".join(c for c in s if not unicodedata.combining(c))
    s = s.lower().strip()
    s = s.replace("&", " and ")
    s = re.sub(r"[\(\)\[\]\{\}]", " ", s)
    s = re.sub(r"[^a-z0-9]+", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

def norm_year(y):
    if pd.isna(y): 
        return pd.NA
    s = str(y).strip()
    # prende i primi 4 digit se presenti
    m = re.search(r"(\d{4})", s)
    return int(m.group(1)) if m else pd.NA

# --- key su metadata: (k_title, k_year) -> row_id ---
if "row_id" not in m.columns:
    m.insert(0, "row_id", range(1, len(m)+1))
    m.to_csv(META_PATH, index=False)
    print("Aggiunto row_id a metadata.csv")

m["k_title"] = m["title"].map(norm_title)
m["k_year"]  = m["release_year"].map(norm_year)

meta_key = m[["row_id","k_title","k_year"]].copy()

# --- definisci qui i match file e quali colonne usare come (title, year) ---
# NB: nel tuo screenshot il manual_overrides usa orig_title / orig_year
MATCH_SPECS = [
    ("data/omdb_matches_manual_overrides.csv",      "orig_title",     "orig_year"),
    ("data/omdb_matches_round3_search.csv",         "orig_title",     "orig_year"),
    ("data/omdb_matches_round2_no_year.csv",        "orig_title",     "orig_year"),
    ("data/omdb_matches_from_variants.csv",         "orig_title",     "orig_year"),
]

out_paths = []

for path, col_t, col_y in MATCH_SPECS:
    if not Path(path).exists():
        print("SKIP (non trovato):", path)
        continue

    df = pd.read_csv(path)

    if "imdb_id" not in df.columns:
        raise ValueError(f"{path} non contiene imdb_id")

    if col_t not in df.columns:
        raise ValueError(f"{path} non contiene la colonna titolo '{col_t}'")
    if col_y not in df.columns:
        # se manca anno, lo settiamo a NA (match più deboli)
        df[col_y] = pd.NA

    df["k_title"] = df[col_t].map(norm_title)
    df["k_year"]  = df[col_y].map(norm_year)

    # merge per portare row_id
    df2 = df.merge(meta_key, on=["k_title","k_year"], how="left")

    # diagnostica rapida
    miss = df2["row_id"].isna().sum()
    print(f"{Path(path).name}: righe={len(df2)} | row_id mancanti dopo merge={miss}")

    out = path.replace(".csv", "_rowid.csv")
    df2.to_csv(out, index=False)
    out_paths.append(out)
    print("SALVATO:", out)

print("Creati:", out_paths)


Aggiunto row_id a metadata.csv
omdb_matches_manual_overrides.csv: righe=75 | row_id mancanti dopo merge=3
SALVATO: data/omdb_matches_manual_overrides_rowid.csv
omdb_matches_round3_search.csv: righe=40 | row_id mancanti dopo merge=0
SALVATO: data/omdb_matches_round3_search_rowid.csv
omdb_matches_round2_no_year.csv: righe=26 | row_id mancanti dopo merge=0
SALVATO: data/omdb_matches_round2_no_year_rowid.csv
omdb_matches_from_variants.csv: righe=87 | row_id mancanti dopo merge=0
SALVATO: data/omdb_matches_from_variants_rowid.csv
Creati: ['data/omdb_matches_manual_overrides_rowid.csv', 'data/omdb_matches_round3_search_rowid.csv', 'data/omdb_matches_round2_no_year_rowid.csv', 'data/omdb_matches_from_variants_rowid.csv']


In [13]:
import pandas as pd
from pathlib import Path

# 1) file in ordine di priorità (il primo vince)
SOURCES = [
    ("manual",   "data/omdb_matches_manual_overrides_rowid.csv", "data/omdb_matches_manual_overrides.csv"),
    ("round3",   "data/omdb_matches_round3_search_rowid.csv",    "data/omdb_matches_round3_search.csv"),
    ("round2",   "data/omdb_matches_round2_no_year_rowid.csv",   "data/omdb_matches_round2_no_year.csv"),
    ("variants", "data/omdb_matches_from_variants_rowid.csv",    "data/omdb_matches_from_variants.csv"),
]

dfs = []
for src, p_rowid, p_fallback in SOURCES:
    path = p_rowid if Path(p_rowid).exists() else p_fallback
    if not Path(path).exists():
        print("SKIP:", path)
        continue

    df = pd.read_csv(path)

    # qui assumiamo che nei tuoi file ci siano almeno queste colonne:
    # row_id, imdb_id
    if "row_id" not in df.columns or "imdb_id" not in df.columns:
        raise ValueError(f"{path} non contiene row_id e imdb_id")

    df = df[["row_id", "imdb_id"]].copy()
    df["source"] = src

    # pulizia base
    df["row_id"] = pd.to_numeric(df["row_id"], errors="coerce").astype("Int64")
    df["imdb_id"] = df["imdb_id"].astype(str).str.strip()
    df = df[df["row_id"].notna() & df["imdb_id"].str.startswith("tt")]

    dfs.append(df)

all_matches = pd.concat(dfs, ignore_index=True)

# 2) PRIORITÀ: tieni il primo imdb_id per ogni row_id (manual > round3 > round2 > variants)
priority = {"manual": 0, "round3": 1, "round2": 2, "variants": 3}
all_matches["prio"] = all_matches["source"].map(priority)

final_map = (
    all_matches.sort_values(["row_id", "prio"])
    .drop_duplicates(subset=["row_id"], keep="first")
    .drop(columns=["prio"])
)

# salva mapping unico
final_map.to_csv("data/imdb_mapping_final.csv", index=False)
print("Creato: data/imdb_mapping_final.csv | righe:", len(final_map))
final_map.head(10)


Creato: data/imdb_mapping_final.csv | righe: 225


,row_id,imdb_id,source
72,1,tt0080684,round3
138,24,tt0080491,variants
139,29,tt0080919,variants
140,71,tt0079501,variants
0,72,tt0080436,manual
1,75,tt0080895,manual
112,81,tt6945338,round2
2,90,tt0080917,manual
3,94,tt0081758,manual
141,102,tt0081573,variants


In [14]:
META_PATH = "data/metadata.csv"
m = pd.read_csv(META_PATH)
mp = pd.read_csv("data/imdb_mapping_final.csv")

# assicurati che row_id sia numerico coerente
m["row_id"] = pd.to_numeric(m["row_id"], errors="coerce").astype("Int64")
mp["row_id"] = pd.to_numeric(mp["row_id"], errors="coerce").astype("Int64")

before_missing = m["imdb_id"].isna().sum() if "imdb_id" in m.columns else len(m)

# merge
out = m.merge(mp, on="row_id", how="left", suffixes=("", "_new"))

# se imdb_id non esiste, crealo
if "imdb_id" not in out.columns:
    out["imdb_id"] = pd.NA

# sovrascrivi imdb_id solo dove la mappa ha un valore
mask = out["imdb_id_new"].notna()
out.loc[mask, "imdb_id"] = out.loc[mask, "imdb_id_new"]

# opzionale: traccia fonte
if "imdb_source" not in out.columns:
    out["imdb_source"] = pd.NA
out.loc[mask, "imdb_source"] = out.loc[mask, "source"]

# pulizia
out = out.drop(columns=["imdb_id_new", "source"], errors="ignore")

after_missing = out["imdb_id"].isna().sum()
print("Missing imdb_id prima:", before_missing)
print("Missing imdb_id dopo :", after_missing)
print("Aggiornati          :", int(mask.sum()))

# sovrascrivi metadata
out.to_csv(META_PATH, index=False)
print("SOVRASCRITTO:", META_PATH)


Missing imdb_id prima: 227
Missing imdb_id dopo : 3
Aggiornati          : 225
SOVRASCRITTO: data/metadata.csv


In [17]:
import pandas as pd

META_PATH = "data/metadata.csv"
MAN_PATH  = "data/omdb_matches_manual_overrides.csv"

m = pd.read_csv(META_PATH)

# ricrea le stesse chiavi che hai usato nella cella precedente
# (se norm_title/norm_year sono già definite sopra, riusale)
import re, unicodedata

def norm_title(s):
    s = "" if pd.isna(s) else str(s)
    s = s.replace("…", "...").replace("’", "'").replace("“","").replace("”","")
    s = unicodedata.normalize("NFKD", s)
    s = "".join(c for c in s if not unicodedata.combining(c))
    s = s.lower().strip()
    s = re.sub(r"[^a-z0-9\s]", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

def norm_year(y):
    if pd.isna(y): return pd.NA
    try: return int(str(y)[:4])
    except: return pd.NA

m["k_title"] = m["title"].map(norm_title)
m["k_year"]  = m["release_year"].map(norm_year)

man = pd.read_csv(MAN_PATH)
man["k_title"] = man["orig_title"].map(norm_title)
man["k_year"]  = man["orig_year"].map(norm_year)

tmp = man.merge(m[["row_id","k_title","k_year","title","release_year"]],
                on=["k_title","k_year"], how="left")

missing = tmp[tmp["row_id"].isna()].copy()
print("Manual overrides non agganciati:", len(missing))
missing[["orig_title","orig_year","imdb_id","canonical_title","canonical_year","note"]].head(20)


Manual overrides non agganciati: 3


,orig_title,orig_year,imdb_id,canonical_title,canonical_year,note
48,A Christmas Carol,2009,tt1067106,NaN,NaN,NaN
61,Cloudy with a Chance of Meatballs 2,2013,tt1985966,NaN,NaN,NaN
63,Planes,2013,tt1691917,NaN,NaN,NaN


In [20]:
import difflib

def suggest_rows(orig_title, orig_year, m, top=10):
    kt = norm_title(orig_title)
    y = norm_year(orig_year)

    # 1) prova stesso anno
    pool = m[m["k_year"].eq(y)].copy()
    if pool.empty:
        pool = m.copy()

    # 2) similarità su k_title
    candidates = difflib.get_close_matches(kt, pool["k_title"].tolist(), n=top, cutoff=0.6)
    out = pool[pool["k_title"].isin(candidates)][["row_id","title","release_year"]].head(top)
    return out

for _, r in missing.iterrows():
    print("\n====", r["orig_title"], r["orig_year"], "->", r["imdb_id"])
    display(suggest_rows(r["orig_title"], r["orig_year"], m, top=12))



==== A Christmas Carol 2009 -> tt1067106


,row_id,title,release_year
2921,2922,Disney’s A Christmas Carol,2009



==== Cloudy with a Chance of Meatballs 2 2013 -> tt1985966


,row_id,title,release_year
3323,3324,loudy with a Chance of Meatballs 2,2013



==== Planes 2013 -> tt1691917


,row_id,title,release_year
3338,3339,Disney Planes,2013


In [4]:
import pandas as pd

fixes = pd.DataFrame([
    {"imdb_id": "tt1067106", "row_id": 2922},
    {"imdb_id": "tt1985966", "row_id": 3324},
    {"imdb_id": "tt1691917", "row_id": 3339},
])

fixes["source"] = "manual_fix"
fixes.to_csv("data/manual_rowid_fixes.csv", index=False)

print("Salvato manual_rowid_fixes.csv - righe:", len(fixes))
display(fixes)


Salvato manual_rowid_fixes.csv - righe: 3


,imdb_id,row_id,source
0,tt1067106,2922,manual_fix
1,tt1985966,3324,manual_fix
2,tt1691917,3339,manual_fix


In [10]:
import pandas as pd

META_PATH = "data/metadata.csv"
MAP_PATH  = "data/manual_rowid_fixes.csv"   # <-- se il tuo file si chiama diverso, cambia qui

# 1) carica
m  = pd.read_csv(META_PATH)
mp = pd.read_csv(MAP_PATH)

# 2) pulizia/typing minimo
m["row_id"]  = pd.to_numeric(m["row_id"], errors="coerce").astype("Int64")
mp["row_id"] = pd.to_numeric(mp["row_id"], errors="coerce").astype("Int64")

mp["imdb_id"] = mp["imdb_id"].astype(str).str.strip()
mp.loc[mp["imdb_id"].isin(["", "nan", "None", "<NA>"]), "imdb_id"] = pd.NA

if "source" not in mp.columns:
    mp["source"] = pd.NA

# tieni solo le colonne utili e rimuovi duplicati row_id (prende la prima occorrenza)
mp = mp[["row_id", "imdb_id", "source"]].dropna(subset=["row_id"])
mp = mp.drop_duplicates(subset=["row_id"], keep="first")

# 3) merge (left: mantieni tutte le righe di metadata)
before_missing = m["imdb_id"].isna().sum() if "imdb_id" in m.columns else len(m)

merged = m.merge(mp, on="row_id", how="left", suffixes=("", "_new"))

# se imdb_id non esiste in metadata, crealo
if "imdb_id" not in merged.columns:
    merged["imdb_id"] = pd.NA

# 4) sovrascrivi imdb_id SOLO dove la mappa ha un valore
mask = merged["imdb_id_new"].notna()
merged.loc[mask, "imdb_id"] = merged.loc[mask, "imdb_id_new"]

# traccia la provenienza (imdb_source)
if "imdb_source" not in merged.columns:
    merged["imdb_source"] = pd.NA
merged.loc[mask, "imdb_source"] = merged.loc[mask, "source"]

# 5) pulizia colonne tecniche
merged = merged.drop(columns=["imdb_id_new", "source"], errors="ignore")

after_missing = merged["imdb_id"].isna().sum()

print("Righe metadata:", len(merged))
print("Missing imdb_id prima:", before_missing)
print("Aggiornati da mappa:", int(mask.sum()))
print("Missing imdb_id dopo:", after_missing)

# 6) salva SOVRASCRIVENDO metadata.csv
merged.to_csv(META_PATH, index=False)
print("SOVRASCRITTO:", META_PATH)

display(merged.head(10))


Righe metadata: 4100
Missing imdb_id prima: 3
Aggiornati da mappa: 3
Missing imdb_id dopo: 0
SOVRASCRITTO: data/metadata.csv


,row_id,rank,imdb_id,title,box_office_year,release_year,distributor,domestic_box_office_gross,tickets_sold,imdb_source
0,1,1,tt0080684,Star Wars Ep. V: The Empire Strikes Back,1980,1980,20th Century Fox,181353855,67417790.0,round3
1,2,2,tt0081562,Stir Crazy,1980,1980,Columbia,101300000,37657992.0,NaN
2,3,3,tt0079417,Kramer vs. Kramer,1980,1979,NaN,98982763,36796566.0,NaN
3,4,4,tt0080339,Airplane!,1980,1980,Paramount Pictures,83453539,31023620.0,NaN
4,5,5,tt0080377,Any Which Way You Can,1980,1980,Warner Bros.,70687344,26277823.0,NaN
5,6,6,tt0081375,Private Benjamin,1980,1980,Warner Bros.,69847348,25965556.0,NaN
6,7,7,tt0080549,Coal Miner's Daughter,1980,1980,Universal,67182787,24975013.0,NaN
7,8,8,tt0081529,Smokey and the Bandit II,1980,1980,Universal,66132626,24584619.0,NaN
8,9,9,tt0080455,The Blues Brothers,1980,1980,Universal,57229890,21275052.0,NaN
9,10,10,tt0081283,Ordinary People,1980,1980,Paramount Pictures,52302978,19443486.0,NaN


In [4]:
import pandas as pd

# =========================
# CONFIG
# =========================
MASTER_PATH = "data/boxoffice_master_domestic.csv"
META_PATH   = "data/metadata.csv"
OUT_PATH    = "data/metadata/metadata_final.csv"

# colonne finali richieste (nomi standardizzati)
FINAL_COLS = [
    "rank",
    "box_office_year",
    "imdb_id",
    "title",
    "release_year",
    "distributor",
    "genre",
    "domestic_box_office_gross",
    "tickets_sold",
]

# =========================
# LOAD
# =========================
master = pd.read_csv(MASTER_PATH)
meta   = pd.read_csv(META_PATH)

print("master shape:", master.shape)
print("meta shape:", meta.shape)

# =========================
# NORMALIZZA NOMI COLONNE (MASTER)
# =========================
master_rename = {}

# rank
if "rank" not in master.columns and "Rank" in master.columns:
    master_rename["Rank"] = "rank"

# box_office_year
if "box_office_year" not in master.columns and "boxoffice_year" in master.columns:
    master_rename["boxoffice_year"] = "box_office_year"

# release_year
if "release_year" not in master.columns and "release_date_year" in master.columns:
    master_rename["release_date_year"] = "release_year"

# domestic gross
if "domestic_box_office_gross" not in master.columns:
    if "domestic_box_office" in master.columns:
        master_rename["domestic_box_office"] = "domestic_box_office_gross"
    elif "domestic_gross" in master.columns:
        master_rename["domestic_gross"] = "domestic_box_office_gross"
    elif "domestic_gross_raw" in master.columns:
        master_rename["domestic_gross_raw"] = "domestic_box_office_gross"

# tickets sold
if "tickets_sold" not in master.columns and "ticket_sold" in master.columns:
    master_rename["ticket_sold"] = "tickets_sold"

master = master.rename(columns=master_rename)

# =========================
# NORMALIZZA NOMI COLONNE (METADATA)
# =========================
meta_rename = {}
if "imdbID" in meta.columns and "imdb_id" not in meta.columns:
    meta_rename["imdbID"] = "imdb_id"
meta = meta.rename(columns=meta_rename)

# =========================
# CREA CHIAVE DI MERGE ROBUSTA
# (usiamo (box_office_year, rank) perché è univoca nel tuo master 1980-2020)
# =========================
need_keys = ["box_office_year", "rank"]
for k in need_keys:
    if k not in master.columns:
        raise ValueError(f"Nel master manca la colonna chiave: {k}")
    if k not in meta.columns:
        raise ValueError(f"Nel metadata manca la colonna chiave: {k} (serve per fare merge con il master)")

# forza tipi
for df in (master, meta):
    df["box_office_year"] = pd.to_numeric(df["box_office_year"], errors="coerce").astype("Int64")
    df["rank"] = pd.to_numeric(df["rank"], errors="coerce").astype("Int64")

# =========================
# PRENDO DAL MASTER SOLO LE COLONNE CHE MI SERVONO
# (così "completano" metadata con genre + domestic gross ecc.)
# =========================
master_needed = ["box_office_year", "rank", "title", "release_year", "distributor", "genre",
                 "domestic_box_office_gross", "tickets_sold"]
master_present = [c for c in master_needed if c in master.columns]
missing_master = [c for c in master_needed if c not in master.columns]
if missing_master:
    print("ATTENZIONE: nel master mancano queste colonne:", missing_master)

master_small = master[master_present].copy()

# =========================
# MERGE: meta (left) + master_small
# =========================
merged = meta.merge(
    master_small,
    on=["box_office_year", "rank"],
    how="left",
    suffixes=("", "_from_master")
)

# Se in metadata esistono già alcune colonne, riempi con i valori del master dove mancano
for c in ["title", "release_year", "distributor", "genre", "domestic_box_office_gross", "tickets_sold"]:
    c_from = f"{c}_from_master"
    if c_from in merged.columns:
        if c in merged.columns:
            merged[c] = merged[c].combine_first(merged[c_from])
            merged = merged.drop(columns=[c_from])
        else:
            merged = merged.rename(columns={c_from: c})

# pulizia imdb_id
if "imdb_id" in merged.columns:
    merged["imdb_id"] = merged["imdb_id"].astype(str).str.strip()
    merged.loc[merged["imdb_id"].isin(["", "nan", "None", "<NA>"]), "imdb_id"] = pd.NA

# =========================
# COSTRUISCI OUTPUT FINALE CON LE 9 COLONNE RICHIESTE
# =========================
present_final = [c for c in FINAL_COLS if c in merged.columns]
missing_final = [c for c in FINAL_COLS if c not in merged.columns]

out = merged[present_final].copy()

# salva
out.to_csv(OUT_PATH, index=False)

print("\nSalvato:", OUT_PATH)
print("Colonne output:", present_final)
print("Colonne mancanti (non trovate dopo merge):", missing_final)
print("Shape output:", out.shape)

display(out.head(20))


master shape: (4100, 12)
meta shape: (4100, 10)

Salvato: data/metadata/metadata_final.csv
Colonne output: ['rank', 'box_office_year', 'imdb_id', 'title', 'release_year', 'distributor', 'genre', 'domestic_box_office_gross', 'tickets_sold']
Colonne mancanti (non trovate dopo merge): []
Shape output: (4100, 9)


,rank,box_office_year,imdb_id,title,release_year,distributor,genre,domestic_box_office_gross,tickets_sold
0,1,1980,tt0080684,Star Wars Ep. V: The Empire Strikes Back,1980,20th Century Fox,Adventure,181353855,67417790.0
1,2,1980,tt0081562,Stir Crazy,1980,Columbia,Black Comedy,101300000,37657992.0
2,3,1980,tt0079417,Kramer vs. Kramer,1979,NaN,Drama,98982763,36796566.0
3,4,1980,tt0080339,Airplane!,1980,Paramount Pictures,Comedy,83453539,31023620.0
4,5,1980,tt0080377,Any Which Way You Can,1980,Warner Bros.,Comedy,70687344,26277823.0
5,6,1980,tt0081375,Private Benjamin,1980,Warner Bros.,Comedy,69847348,25965556.0
6,7,1980,tt0080549,Coal Miner's Daughter,1980,Universal,Drama,67182787,24975013.0
7,8,1980,tt0081529,Smokey and the Bandit II,1980,Universal,Comedy,66132626,24584619.0
8,9,1980,tt0080455,The Blues Brothers,1980,Universal,Comedy,57229890,21275052.0
9,10,1980,tt0081283,Ordinary People,1980,Paramount Pictures,Drama,52302978,19443486.0


<a style='text-decoration:none;line-height:16px;display:flex;color:#5B5B62;padding:10px;justify-content:end;' href='https://deepnote.com?utm_source=created-in-deepnote-cell&projectId=e08cfdf8-9b6e-44e8-b36c-3dd235d85ba1' target="_blank">

Created in <span style='font-weight:600;margin-left:4px;'>Deepnote</span></a>